# Fate marker example

Load a single organoid mesh, show **all cell fates** on its surface, then look at the
heat-kernel signature (HKS), the fate / HKS correlation, and the bag-of-features encoding.
Fate fields are refined with `FateMarkers._refine_markers`, which combines multi-channel
markers (e.g. `ta` = cycd or cyca) and applies the biological mutual-exclusion hierarchy
defined in `config.json` (matching the CSV's `*.cnt_exclusive` definition).

In [ ]:
import json

import igl
import numpy as np
import scipy.sparse as sp
from meshplot import plot
from matplotlib import pyplot as plt

from src.fatemarkers import FateMarkers

In [ ]:
folder_path = 'Data/20260224/'
timepoint = 'day4p5'

with open(f"{folder_path}config.json") as f:
    cfg = json.load(f)

annotation_names = cfg['annotation_names']     # friendly name -> vtp field(s)
exclusion_rules  = cfg['exclusion_rules']      # biological mutual-exclusion hierarchy
fate_order       = list(annotation_names.keys())

zarr_name  = cfg['zarr_names'][timepoint]
round_name = cfg['rounds'][timepoint]
mesh_name  = cfg['mesh_name']

## Load an organoid mesh

`_refine_markers` runs right after loading (before PCA / eigendecomposition), exactly as in the
`run_new_meshes.py` pipeline, so the fields and coefficients here match the saved ones.

In [ ]:
label = 'day4p5_B06_63'.split('_')
path = (f'{folder_path}fractal_output/{label[0]}/{zarr_name}/{label[1][0]}/{label[1][1:]}/'
        f'{round_name}/meshes/{mesh_name}/{label[2]}.vtp')

m = FateMarkers()
m.load_mesh_from_file(path)
m._refine_markers(annotation_names, exclusion_rules)   # combine 'ta' channels + mutual exclusion
m.align_with_pca()
m.precompute_eigens()
m.compute_coefficients()

print('area:', m.area)
print('vertices:', m.v.shape[0], '| faces:', m.f.shape[0])

## Show all cell fates

Each fate marker, in the canonical `fate_order`, plotted on the mesh surface. `ta` is the combined
cell-cycle channel appended by `_refine_markers`.

In [ ]:
# resolve each fate to its field column (list-valued markers, e.g. 'ta', are
# combined by _refine_markers into a single field named after the marker)
def fate_index(name):
    fld = annotation_names[name]
    return m.field_names.index(name if isinstance(fld, list) else fld)

for name in fate_order:
    idx = fate_index(name)
    vals = m.fields[:, idx]
    print(f'{name:8s}  positive vertices: {int((vals > 0).sum()):6d}')
    plot(m.v, m.f, c=vals)

## Heat kernel signature (HKS)

HKS at a series of diffusion times (time ~ distance squared); a larger time probes a larger
neighbourhood around each vertex.

In [ ]:
ts = [1, 4, 25, 100]
hks = m.compute_hks_for_new_times(ts, coeffs=False)   # (n_vertices, len(ts))
print('hks shape:', hks.shape)

for k, t in enumerate(ts):
    print(f'HKS at t={t}')
    plot(m.v, m.f, c=hks[:, k])

## Correlation between fate markers and HKS

Spherical-harmonic coefficients of the fate fields and of the HKS fields, compared via a
mass-weighted correlation (ignoring the degree-0 / mean term).

In [ ]:
coeffs_fm  = m.coeffs_fm[:, [fate_index(n) for n in fate_order]]
hks_coeffs = m.compute_hks_for_new_times(ts, coeffs=True)
coeffs = np.concatenate([coeffs_fm, hks_coeffs], axis=1)

labels = fate_order + [f'HKS t={t}' for t in ts]
area = m.mass_matrix.diagonal().sum()

corr_matrix = np.zeros((coeffs.shape[1], coeffs.shape[1]))
for i in range(coeffs.shape[1]):
    for j in range(i + 1, coeffs.shape[1]):
        corr  = np.sum(coeffs[1:, i] * coeffs[1:, j]) / area
        var_i = np.sum(coeffs[1:, i] ** 2) / area
        var_j = np.sum(coeffs[1:, j] ** 2) / area
        corr_matrix[i, j] = corr / np.sqrt(var_i * var_j)

plt.imshow(corr_matrix.T, cmap='bwr', vmin=-0.2, vmax=0.2)
plt.colorbar()
plt.xticks(np.arange(len(labels)), labels, rotation=90)
plt.yticks(np.arange(len(labels)), labels)
plt.title('Correlation: fate markers and HKS')
plt.show()

## Bag-of-features encoding

Soft-assign each vertex's multi-scale HKS curve to a learned vocabulary of shape "words".

In [ ]:
res    = np.load('sim/vocab_new.npz', allow_pickle=True)
vocab  = res['vocab']
scaler = res['scaler'].item()
sigma  = res['sigma']
vocab_ts = res['ts']

hks_fine = m.compute_hks_for_new_times(vocab_ts, coeffs=False)
normalised_hks = scaler.transform(hks_fine / np.mean(hks_fine, axis=0, keepdims=True) - 1)

# soft-assignment of each vertex to each vocab word
dist = np.linalg.norm(normalised_hks[:, None, :] - vocab[None, :, :], axis=2)
encoding = np.exp(-dist ** 2 / (2 * sigma ** 2))     # (n_vertices, n_words)
print('encoding shape:', encoding.shape)

In [ ]:
from matplotlib.colors import Normalize
from matplotlib import cm

random_indices = np.random.choice(hks_fine.shape[0], size=100, replace=False)
norm = Normalize(vmin=hks_fine[:, 0].min(), vmax=hks_fine[:, 0].max())
cmap = cm.get_cmap('viridis')

fig, ax = plt.subplots(figsize=(7, 5))
for ind in random_indices:
    ax.plot(normalised_hks[ind], color=cmap(norm(hks_fine[ind, 0])), alpha=0.5)
for i, center in enumerate(vocab):
    ax.plot(center, '-', linewidth=2, label=f'word {i}')
ax.set_xlabel('HKS time-scale index'); ax.set_ylabel('normalised HKS')
plt.legend(loc='upper right', bbox_to_anchor=(1.18, 1), fontsize=10)
plt.show()

In [ ]:
# each bag-of-features word visualised on the mesh
for i in range(encoding.shape[1]):
    print(f'word {i}:  [{encoding[:, i].min():.3f}, {encoding[:, i].max():.3f}]')
    plot(m.v, m.f, c=encoding[:, i])